# Extração de clientes e envio ao n8n

O notebook lê a tabela `#clientes` publicada no GitHub Pages. Se a página não estiver acessível ou não contiver clientes, usa `docs/index.html` local. Configure `N8N_WEBHOOK_URL` no ambiente antes de executar a última célula.

In [ ]:
import logging
import os
import re
from pathlib import Path

import requests
from bs4 import BeautifulSoup

URL_PAGINA = os.getenv(
    "CLIENTES_URL",
    "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/",
)
N8N_WEBHOOK_URL = os.getenv("N8N_WEBHOOK_URL", "").strip()
TIMEOUT_SEGUNDOS = 20

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("rpa_investimentos")


def converter_saldo(valor):
    """Converte valores brasileiros como 'R$ 12.500,00' para número JSON."""
    normalizado = re.sub(r"[^0-9,.-]", "", valor).replace(".", "").replace(",", ".")
    try:
        return float(normalizado)
    except ValueError as erro:
        raise ValueError(f"Saldo inválido na tabela: {valor!r}") from erro


def extrair_clientes(html):
    """Extrai campos pelos seletores da tabela existente em docs/index.html."""
    soup = BeautifulSoup(html, "html.parser")
    clientes = []
    for linha in soup.select("#clientes tbody tr"):
        campos = {
            nome: (linha.select_one(f".{nome}").get_text(" ", strip=True)
                   if linha.select_one(f".{nome}") else "")
            for nome in ("nome", "email", "saldo", "perfil")
        }
        if not all(campos.values()):
            raise ValueError("Linha de cliente sem um ou mais campos obrigatórios.")
        clientes.append({
            "nome": campos["nome"],
            "email": campos["email"],
            "saldo": converter_saldo(campos["saldo"]),
            "perfil": campos["perfil"],
        })
    return clientes


def carregar_html_local():
    """Procura a página local tanto na raiz do repositório quanto em rpa/."""
    candidatos = (Path.cwd() / "docs" / "index.html", Path.cwd().parent / "docs" / "index.html")
    for caminho in candidatos:
        if caminho.is_file():
            logger.info("Usando a página local: %s", caminho)
            return caminho.read_text(encoding="utf-8")
    raise FileNotFoundError("Não foi possível acessar a página remota nem localizar docs/index.html.")


def obter_clientes():
    """Prefere a página publicada e recorre ao HTML local quando necessário."""
    try:
        resposta = requests.get(URL_PAGINA, timeout=TIMEOUT_SEGUNDOS)
        resposta.raise_for_status()
        clientes = extrair_clientes(resposta.text)
        if clientes:
            logger.info("Extraídos %d clientes da página publicada.", len(clientes))
            return clientes
        logger.warning("A página publicada não contém clientes; tentando o arquivo local.")
    except (requests.RequestException, ValueError) as erro:
        logger.warning("Falha ao carregar clientes publicados (%s); tentando o arquivo local.", erro)
    clientes = extrair_clientes(carregar_html_local())
    if not clientes:
        raise ValueError("Nenhum cliente foi encontrado na página local.")
    logger.info("Extraídos %d clientes do arquivo local.", len(clientes))
    return clientes


def enviar_para_n8n(clientes):
    """Envia a lista estruturada e reporta erros HTTP/rede ao operador."""
    if not N8N_WEBHOOK_URL:
        raise RuntimeError("Configure a variável de ambiente N8N_WEBHOOK_URL antes de enviar os dados.")
    try:
        resposta = requests.post(
            N8N_WEBHOOK_URL,
            json={"clientes": clientes},
            headers={"Content-Type": "application/json"},
            timeout=TIMEOUT_SEGUNDOS,
        )
        resposta.raise_for_status()
    except requests.RequestException as erro:
        logger.exception("Falha ao enviar os clientes ao webhook do n8n.")
        raise RuntimeError("O envio ao webhook do n8n falhou.") from erro
    logger.info("Webhook respondeu com HTTP %s.", resposta.status_code)
    return resposta


In [ ]:
clientes = obter_clientes()
logger.info("Clientes encontrados: %s", clientes)
resposta = enviar_para_n8n(clientes)
print(f"Envio concluído: {len(clientes)} clientes; HTTP {resposta.status_code}.")